# 41 Analysis — Guest Catalogue and Property Distribution

Builds a flat guest catalogue from Phase 32 deduplication output joined with Wikidata property data.
Computes page-rank on the Wikidata graph to find the most central persons.

**Prerequisite:** Phase 32 must have been run (`dedup_persons.csv` must exist).

**Outputs:** `data/40_analysis/`
- `guest_catalogue.csv` — flat table of Wikidata-matched persons with properties
- `pagerank_persons.csv` — page-rank scores for all person nodes
- `analysis_summary.json` — key statistics

In [ ]:
import sys
import pathlib

_src = pathlib.Path('.').resolve().parent
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import json
import pandas as pd
from collections import Counter
from datetime import datetime, timezone

OUTPUT_DIR = pathlib.Path('data/40_analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from process.io_guardrails import atomic_write_csv, atomic_write_text

## Stage 4a: Load data and build guest catalogue

In [ ]:
dedup_persons = pd.read_csv('data/32_entity_deduplication/dedup_persons.csv', dtype=str).fillna('')
wd_persons = dedup_persons[dedup_persons['cluster_strategy'] == 'wikidata_qid_match'].copy()
print(f'Wikidata-matched canonical entities: {len(wd_persons)}')

inst = pd.read_csv(
    'data/20_candidate_generation/wikidata/projections/instances.csv', dtype=str
).fillna('')
qid_label = dict(zip(inst['qid'], inst['labels_de'].where(inst['labels_de'] != '', inst['label'])))

persons_json = json.loads(
    pathlib.Path('data/20_candidate_generation/wikidata/projections/instances_core_persons.json')
    .read_text(encoding='utf-8')
)
print(f'Wikidata persons with claim data: {len(persons_json)}')

In [ ]:
GENDER_QID = {
    'Q6581097': 'male',
    'Q6581072': 'female',
    'Q1097630': 'intersex',
    'Q48270': 'non-binary',
    'Q2449503': 'transgender male',
    'Q1052281': 'transgender female',
}


def extract_item_qid(stmt):
    try:
        return stmt['mainsnak']['datavalue']['value']['id']
    except (KeyError, TypeError):
        return ''


def extract_time_year(stmt):
    try:
        t = stmt['mainsnak']['datavalue']['value']['time']
        return t[1:5]
    except (KeyError, TypeError):
        return ''


catalogue_rows = []
for _, person_row in wd_persons.iterrows():
    qid = person_row['wikidata_id']
    entity = persons_json.get(qid, {})
    claims = entity.get('claims', {})
    label_de = entity.get('labels', {}).get('de', {}).get('value', '')
    label_en = entity.get('labels', {}).get('en', {}).get('value', '')
    gender_qid = extract_item_qid(claims['P21'][0]) if 'P21' in claims else ''
    gender = GENDER_QID.get(gender_qid, '')
    occ_qids = [extract_item_qid(s) for s in claims.get('P106', [])]
    occ_labels = [qid_label.get(q, q) for q in occ_qids if q]
    party_qids = [extract_item_qid(s) for s in claims.get('P102', [])]
    party_labels = [qid_label.get(q, q) for q in party_qids if q]
    employer_qids = [extract_item_qid(s) for s in claims.get('P108', [])]
    employer_labels = [qid_label.get(q, q) for q in employer_qids if q]
    birthyear = extract_time_year(claims['P569'][0]) if 'P569' in claims else ''
    catalogue_rows.append({
        'canonical_entity_id': person_row['canonical_entity_id'],
        'wikidata_id': qid,
        'cluster_size': int(person_row['cluster_size']),
        'label_de': label_de,
        'label_en': label_en,
        'gender_qid': gender_qid,
        'gender': gender,
        'occupations': '|'.join(occ_labels),
        'occupation_qids': '|'.join(occ_qids),
        'party': '|'.join(party_labels),
        'employer': '|'.join(employer_labels),
        'birthyear': birthyear,
    })

catalogue = pd.DataFrame(catalogue_rows)
atomic_write_csv(OUTPUT_DIR / 'guest_catalogue.csv', catalogue)
print(f'Guest catalogue: {len(catalogue)} persons -> saved to data/40_analysis/guest_catalogue.csv')
catalogue.head(5)

## Stage 4b: Property distribution analysis

In [ ]:
total = len(catalogue)

print('=== GENDER DISTRIBUTION (unique persons) ===')
gender_counts = catalogue['gender'].value_counts()
for g, n in gender_counts.items():
    print(f'  {(g or "(unknown)"):20s} {n:5d}  ({100*n/total:.1f}%)')

print()
print('=== GENDER BY APPEARANCE COUNT (weighted by cluster_size) ===')
gender_appearances = catalogue.groupby('gender')['cluster_size'].sum().sort_values(ascending=False)
total_appearances = gender_appearances.sum()
for g, n in gender_appearances.items():
    print(f'  {(g or "(unknown)"):20s} {n:6d}  ({100*n/total_appearances:.1f}%)')

In [ ]:
all_occs = [o for occs in catalogue['occupations'].str.split('|') for o in occs if o]

print('=== TOP OCCUPATIONS (by person count) ===')
for occ, count in Counter(all_occs).most_common(20):
    print(f'  {count:4d}  {occ}')

print()
all_parties = [p for parties in catalogue['party'].str.split('|') for p in parties if p]
print('=== TOP PARTIES (by person count) ===')
for party, count in Counter(all_parties).most_common(15):
    print(f'  {count:4d}  {party}')

## Stage 4c: Page-rank on Wikidata graph

In [ ]:
import networkx as nx

triples = pd.read_csv(
    'data/20_candidate_generation/wikidata/projections/triples.csv', dtype=str
)

# Exclude taxonomy predicates that create class-hub dominance
EXCLUDE_PREDICATES = {'P31', 'P279', 'P1889', 'P460', 'P1269', 'P910', 'P1343', 'P5030', 'P5008', 'P1552', 'P1001'}
filtered = triples[~triples['predicate'].isin(EXCLUDE_PREDICATES)]
print(f'Triples used: {len(filtered)} of {len(triples)} (excluded: {len(triples) - len(filtered)})')

G = nx.DiGraph()
for _, row in filtered.iterrows():
    G.add_edge(row['subject'], row['object'], predicate=row['predicate'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
pagerank = nx.pagerank(G, alpha=0.85)
print('Page-rank computation complete.')

In [ ]:
person_qids = set(persons_json.keys())
person_pr_rows = []
for qid, score in pagerank.items():
    if qid in person_qids:
        entity = persons_json[qid]
        label = entity.get('labels', {}).get('de', {}).get('value', qid_label.get(qid, qid))
        person_pr_rows.append({'wikidata_id': qid, 'label_de': label, 'pagerank': round(score, 8)})

pr_df = pd.DataFrame(person_pr_rows).sort_values('pagerank', ascending=False).reset_index(drop=True)
atomic_write_csv(OUTPUT_DIR / 'pagerank_persons.csv', pr_df)

print('Top 20 persons by page-rank:')
print(pr_df.head(20).to_string(index=False))

In [ ]:
gender_pct = (catalogue['gender'].value_counts() / len(catalogue) * 100).round(1).to_dict()
gender_app_pct = {
    g: round(100 * n / total_appearances, 1)
    for g, n in gender_appearances.items()
}

summary = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'wikidata_matched_persons': total,
    'gender_distribution_pct': gender_pct,
    'gender_by_appearance_pct': gender_app_pct,
    'top_occupations': [occ for occ, _ in Counter(all_occs).most_common(10)],
    'top_person_pagerank': pr_df.head(10)[['wikidata_id', 'label_de', 'pagerank']].to_dict('records'),
}

atomic_write_text(
    OUTPUT_DIR / 'analysis_summary.json',
    json.dumps(summary, indent=2, ensure_ascii=False)
)
print('Stage 4 complete. Outputs in data/40_analysis/')